# Модель на CoBaLD

In [1]:
pip install pyconll

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
import pyconll

from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
from transformers import BertModel

In [5]:
class CustomCoNLLDataset(Dataset):
    def __init__(self, conllu_file, tokenizer, max_length=128, target_column=-1):
        self.data, self.labels = [], set()
        current_sentence, current_labels = [], []
        with open(conllu_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    if not line and current_sentence:
                        self.data.append((current_sentence.copy(), current_labels.copy()))
                        current_sentence, current_labels = [], []
                    continue
                parts = line.split('\t')
                if parts[0].isdigit() or '-' in parts[0]:
                    word = parts[1]
                    sem_class = parts[target_column] if len(parts) > 10 else 'O'
                    current_sentence.append(word)
                    current_labels.append(sem_class)
                    self.labels.add(sem_class)
            if current_sentence:
                self.data.append((current_sentence, current_labels))

        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label2id = {l:i for i,l in enumerate(sorted(self.labels))}
        self.id2label = {i:l for l,i in self.label2id.items()}

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        tokens, labels = self.data[idx]
        text = ' '.join(tokens)
        encoding = self.tokenizer(text, truncation=True, padding='max_length',
                                  max_length=self.max_length, return_tensors='pt')
        bert_tokens = self.tokenizer.convert_ids_to_tokens(encoding['input_ids'][0])
        token_labels = torch.ones(self.max_length, dtype=torch.long) * -100
        lbl_idx = 0
        for i, token in enumerate(bert_tokens):
            if token.startswith('##'):
                if i>0 and token_labels[i-1]!=-100:
                    token_labels[i] = token_labels[i-1]
            elif token in ['[CLS]','[SEP]','[PAD]']:
                continue
            elif lbl_idx < len(labels):
                token_labels[i] = self.label2id.get(labels[lbl_idx],0)
                lbl_idx +=1
        return {'input_ids':encoding['input_ids'].squeeze(),
                'attention_mask':encoding['attention_mask'].squeeze(),
                'labels':token_labels}

class SemanticModel(nn.Module):
    def __init__(self, bert_model, num_labels):
        super().__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(bert_model.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq_out = self.dropout(outputs.last_hidden_state)
        logits = self.classifier(seq_out)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.view(-1, logits.shape[-1]), labels.view(-1))
            return {'loss':loss, 'logits':logits}
        return {'logits':logits}

'''def train_semantic(conllu_file, model_name='DeepPavlov/rubert-base-cased', device=None, epochs=2):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    bert = AutoModel.from_pretrained(model_name)
    ds = CustomCoNLLDataset(conllu_file, tokenizer)
    dl = DataLoader(ds, batch_size=64, shuffle=True)
    model = SemanticModel(bert, len(ds.label2id)).to(device)
    optim = AdamW(model.parameters(), lr=2e-5)
    model.train()
    for epoch in range(epochs):
        total=0
        for b in dl:
            optim.zero_grad()
            inp=b['input_ids'].to(device); att=b['attention_mask'].to(device); lbl=b['labels'].to(device)
            out=model(inp, att, lbl)
            out['loss'].backward(); optim.step()
            total+=out['loss'].item()
        print(f"Semantic Epoch {epoch+1}, Loss={total/len(dl):.4f}")
    return model, tokenizer, ds.id2label'''


def train_and_eval_semantic(conllu_train,
                            conllu_dev,
                            model_name='DeepPavlov/rubert-base-cased',
                            device=None,
                            epochs=2):
    device    = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    bert      = AutoModel.from_pretrained(model_name)

    # Тренировочный датасет
    train_ds = CustomCoNLLDataset(conllu_train, tokenizer)
    train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)

    # Валидационный датасет
    dev_ds = CustomCoNLLDataset(conllu_dev, tokenizer)
    dev_dl = DataLoader(dev_ds, batch_size=256, shuffle=False)

    # Модель и оптимизатор
    model     = SemanticModel(bert, len(train_ds.label2id)).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    for epoch in range(epochs):
        # — тренировка
        model.train()
        total_loss = 0
        for b in train_dl:
            optimizer.zero_grad()
            inp, att, lbl = b['input_ids'].to(device), b['attention_mask'].to(device), b['labels'].to(device)
            out = model(inp, att, lbl)
            out['loss'].backward()
            optimizer.step()
            total_loss += out['loss'].item()
        print(f"[Train] Epoch {epoch+1}, Loss={total_loss/len(train_dl):.4f}")

        # валидация на деве
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for b in dev_dl:
                inp, att = b['input_ids'].to(device), b['attention_mask'].to(device)
                lbl = b['labels'].view(-1).cpu().numpy()
                logits = model(inp, att)['logits']       # [B, L, C]
                preds  = logits.argmax(-1).view(-1).cpu().numpy()
                mask   = lbl != -100
                all_preds.extend(preds[mask])
                all_labels.extend(lbl[mask])

        acc = accuracy_score(all_labels, all_preds)
        f1  = f1_score(all_labels, all_preds, average='macro')
        print(f"[Dev]   Epoch {epoch+1}, Acc={acc:.4f}, F1={f1:.4f}")

    return model, tokenizer, train_ds.label2id


'''class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, sem_model, id2label, max_length=128, device=None):
        self.texts = df['text'].tolist()
        self.labels = df['sarcasm'].tolist()
        self.tokenizer = tokenizer
        self.sem_model = sem_model.to(device)
        self.id2label = id2label
        self.max_length = max_length
        self.device = device
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text=self.texts[idx]
        enc=self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        input_ids, att = enc['input_ids'].squeeze(), enc['attention_mask'].squeeze()
        with torch.no_grad():
            sem_out=self.sem_model(input_ids=input_ids.unsqueeze(0).to(self.device),
                                   attention_mask=att.unsqueeze(0).to(self.device))['logits']
        sem_feats = sem_out.squeeze().cpu()
        return {'input_ids':input_ids,'attention_mask':att,
                'sem_feats':sem_feats,'label':torch.tensor(self.labels[idx],dtype=torch.long)}'''

class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts       = df['text'].tolist()
        self.labels      = df['sarcasm'].tolist()
        #self.sem_pooled  = df['sem_pooled'].tolist()
        self.sem_feats = df['sem_pooled'].tolist()
        self.tokenizer   = tokenizer
        self.max_length  = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids':     enc['input_ids'].squeeze(0),
            'attention_mask':enc['attention_mask'].squeeze(0),
            'sem_feats':    torch.tensor(self.sem_feats[idx], dtype=torch.float),
            'labels':       torch.tensor(self.labels[idx], dtype=torch.long),
        }

'''class SarcasmClassifier(nn.Module):
    def __init__(self, bert_model, sem_dim, num_classes):
        super().__init__()
        self.bert = bert_model
        self.sem_proj = nn.Linear(sem_dim, bert_model.config.hidden_size)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(bert_model.config.hidden_size*2, num_classes)
    def forward(self, input_ids, attention_mask, sem_feats, labels=None):
        bert_out=self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        sem_pooled = sem_feats.mean(dim=1)
        sem_proj = self.sem_proj(sem_pooled)
        joint = torch.cat([bert_out, sem_proj], dim=1)
        joint = self.dropout(joint)
        logits = self.classifier(joint)
        loss=None
        if labels is not None:
            loss_fct=nn.CrossEntropyLoss()
            loss=loss_fct(logits, labels)
            return {'loss':loss,'logits':logits}
        return logits'''

class SarcasmClassifier(nn.Module):
    def __init__(self, bert_model, sem_dim, num_classes):
        super().__init__()
        self.bert     = bert_model
        self.sem_proj = nn.Linear(sem_dim, bert_model.config.hidden_size)
        self.dropout  = nn.Dropout(0.1)
        self.classifier = nn.Linear(bert_model.config.hidden_size*2, num_classes)

    def forward(self, input_ids, attention_mask, sem_feats, labels=None):
        bert_out   = self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        sem_proj   = self.sem_proj(sem_feats)
        joint      = torch.cat([bert_out, sem_proj], dim=1)
        joint      = self.dropout(joint)
        logits     = self.classifier(joint)

        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            return {'loss': loss, 'logits': logits}
        return {'logits': logits}

# Устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Тренировка семантической модели
train_conllu = r'/content/train.conllu'
dev_conllu = r'/content/dev.conllu'
#sem_model, tokenizer, id2label = train_semantic(train_conllu, device=device)
sem_model, tokenizer, id2label = train_and_eval_semantic(
    conllu_train = train_conllu,
    conllu_dev = dev_conllu,
    model_name ='DeepPavlov/rubert-base-cased',
    device = device,
    epochs = 2
)

sem_model.eval()

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[Train] Epoch 1, Loss=1.4142
[Dev]   Epoch 1, Acc=0.0135, F1=0.0112
[Train] Epoch 2, Loss=0.5651
[Dev]   Epoch 2, Acc=0.0149, F1=0.0170


SemanticModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

In [6]:
# Загружаем сырые данные
df = pd.read_csv('/content/dataset_all_data (2).csv')
df['sarcasm'] = df['sarcasm'].astype(int)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Пред–вычисление sem_pooled
sem_model.eval().to(device)
sem_pooled_list = []
for text in df['text'].tolist():
    enc = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        logits = sem_model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask']
        )['logits']           # [1, seq_len, sem_dim]
    pooled = logits.squeeze(0).mean(dim=0).cpu().numpy()  # [sem_dim]
    sem_pooled_list.append(pooled)

df['sem_pooled'] = sem_pooled_list

# DataLoader
ds = SarcasmDataset(df, tokenizer, max_length=128)
dl = DataLoader(ds, batch_size=16, shuffle=True)

# Создаём и обучаем SarcasmClassifier
bert = sem_model.bert
sarcasm_model = SarcasmClassifier(
    bert_model=bert,
    sem_dim=df['sem_pooled'].iloc[0].shape[0],
    num_classes=2
).to(device)

optimizer = AdamW(sarcasm_model.parameters(), lr=2e-5)

sarcasm_model.train()
for epoch in range(3):
    total_loss = 0.0
    for batch in dl:
        optimizer.zero_grad()
        out = sarcasm_model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            sem_feats=batch['sem_feats'].to(device),
            labels=batch['labels'].to(device)
        )
        out['loss'].backward()
        optimizer.step()
        total_loss += out['loss'].item()
    print(f"Sarcasm Epoch {epoch+1}, Loss={total_loss/len(dl):.4f}")

# Сохраняем модель
torch.save(sarcasm_model.state_dict(), 'sarcasm_model.pt')
print("Training complete.")

Sarcasm Epoch 1, Loss=0.4355
Sarcasm Epoch 2, Loss=0.3570
Sarcasm Epoch 3, Loss=0.2271
Training complete.


In [7]:
'''# Загрузка датасета сарказма
df = pd.read_csv(r'/content/dataset_all_data (2).csv')
#ds = SarcasmDataset(df, tokenizer, sem_model, id2label, device=device)
ds = SarcasmDataset(df, tokenizer, max_length=128)
dl = DataLoader(ds, batch_size=16, shuffle=True)

# Создание и тренировка классификатора сарказма
bert = sem_model.bert
sarcasm_model = SarcasmClassifier(bert, sem_dim=len(id2label), num_classes=2).to(device)
optimizer = AdamW(sarcasm_model.parameters(), lr=2e-5)'''

"# Загрузка датасета сарказма\ndf = pd.read_csv(r'/content/dataset_all_data (2).csv')\n#ds = SarcasmDataset(df, tokenizer, sem_model, id2label, device=device)\nds = SarcasmDataset(df, tokenizer, max_length=128)\ndl = DataLoader(ds, batch_size=16, shuffle=True)\n\n# Создание и тренировка классификатора сарказма\nbert = sem_model.bert\nsarcasm_model = SarcasmClassifier(bert, sem_dim=len(id2label), num_classes=2).to(device)\noptimizer = AdamW(sarcasm_model.parameters(), lr=2e-5)"

In [8]:
'''sarcasm_model.train()
for epoch in range(3):
    total_loss = 0
    for batch in dl:
        optimizer.zero_grad()
        out = sarcasm_model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device),
            batch['sem_feats'].to(device),
            batch['label'].to(device)
        )
        out['loss'].backward()
        optimizer.step()
        total_loss += out['loss'].item()
    print(f"Sarcasm Epoch {epoch+1}, Loss={total_loss/len(dl):.4f}")

# Сохранение модели
torch.save(sarcasm_model.state_dict(), 'sarcasm_model.pt')
print("Training complete.")'''

'sarcasm_model.train()\nfor epoch in range(3):\n    total_loss = 0\n    for batch in dl:\n        optimizer.zero_grad()\n        out = sarcasm_model(\n            batch[\'input_ids\'].to(device),\n            batch[\'attention_mask\'].to(device),\n            batch[\'sem_feats\'].to(device),\n            batch[\'label\'].to(device)\n        )\n        out[\'loss\'].backward()\n        optimizer.step()\n        total_loss += out[\'loss\'].item()\n    print(f"Sarcasm Epoch {epoch+1}, Loss={total_loss/len(dl):.4f}")\n\n# Сохранение модели\ntorch.save(sarcasm_model.state_dict(), \'sarcasm_model.pt\')\nprint("Training complete.")'

In [9]:
'''import torch
import numpy as np
import pandas as pd

# Устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Список текстов
texts = df['text'].tolist()

# Генерация усреднённых семантических векторов
sem_pooled_list = []
sem_model.to(device)
for text in texts:
    # Токенизация
    enc = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    ).to(device)

    # Предсказание логитов
    with torch.no_grad():
        logits = sem_model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask']
        )['logits']

    # Усреднение по seq_len = вектор
    pooled = logits.squeeze(0).mean(dim=0).cpu().numpy()
    sem_pooled_list.append(pooled)

df['sem_pooled'] = sem_pooled_list
df.to_pickle('enriched_sarcasm_pooled.pkl')
print(f"Сохранено {len(df)} записей в 'enriched_sarcasm_pooled.pkl' вот так")'''

'import torch\nimport numpy as np\nimport pandas as pd\n\n# Устройство\ndevice = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n\n# Список текстов\ntexts = df[\'text\'].tolist()\n\n# Генерация усреднённых семантических векторов\nsem_pooled_list = []\nsem_model.to(device)\nfor text in texts:\n    # Токенизация\n    enc = tokenizer(\n        text,\n        truncation=True,\n        padding=\'max_length\',\n        max_length=128,\n        return_tensors=\'pt\'\n    ).to(device)\n\n    # Предсказание логитов\n    with torch.no_grad():\n        logits = sem_model(\n            input_ids=enc[\'input_ids\'],\n            attention_mask=enc[\'attention_mask\']\n        )[\'logits\']\n\n    # Усреднение по seq_len = вектор\n    pooled = logits.squeeze(0).mean(dim=0).cpu().numpy()\n    sem_pooled_list.append(pooled)\n\ndf[\'sem_pooled\'] = sem_pooled_list\ndf.to_pickle(\'enriched_sarcasm_pooled.pkl\')\nprint(f"Сохранено {len(df)} записей в \'enriched_sarcasm_pooled.pkl\' вот 

In [10]:
# Генерация sem_pooled для всех текстов — выполняем один раз
sem_pooled_list = []
sem_model.eval()
for text in df['text'].tolist():
    enc = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        logits = sem_model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask']
        )['logits']  # [1, seq_len, num_labels]

    # Усредняем по длине последовательности
    pooled = logits.squeeze(0).mean(dim=0).cpu().numpy()  # [num_labels]
    sem_pooled_list.append(pooled)

# Сохраняем в df и на диск
df['sem_pooled'] = sem_pooled_list
df.to_pickle('enriched_sarcasm_pooled.pkl')
print("Pre-computed semantic features for", len(df), "texts.")

Pre-computed semantic features for 9144 texts.


In [11]:
abc = pd.read_pickle(r'/content/enriched_sarcasm_pooled.pkl')

In [12]:
abc

,text,gender,age,sarcasm,sem_pooled
0,Быстро работает паблик. Его уже вылечили и он ...,2,34,0,"[-0.16029009, -0.9576734, -0.47864425, 0.07563..."
1,"это был я ,если вы о комментариях.А вы прям са...",2,43,1,"[0.32103986, -1.2857925, -0.6879766, -0.745767..."
2,"Георгий, вам начислено 50 + к вашей карме. Ско...",2,43,1,"[1.4222873, -0.90399987, -0.10063143, -1.22621..."
3,"Вы рачьё поганое , тот, кто проигнорировал ком...",2,29,0,"[-0.30396065, -0.7146007, -0.55393815, -0.3882..."
4,вы плять гоните пластик заваривать?по возможно...,2,43,0,"[-1.0845735, -0.504058, -0.25592172, 0.2241364..."
...,...,...,...,...,...
9139,Спасибо золотце))))) и ты мне тем же))))) и те...,1,17,0,"[-0.6515293, -0.67500985, -0.4266207, -0.24721..."
9140,"Игорь, вы просто супер поработали!!!!С вами бы...",1,24,0,"[-0.5627828, -0.7193165, 0.0783171, 0.5083995,..."
9141,Мне (лифтинг ампулы под глаза 2 шт) и куда опл...,1,35,0,"[-1.099153, -0.5120387, -0.3653117, 0.10331926..."
9142,В Рязани научились коптить грудинку? Не-ве-рю!...,2,40,0,"[-0.69003654, -0.98125786, -0.12811628, -0.662..."


## Обучение

In [13]:
import random

In [14]:
def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_random_seed(12345)

In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro')
    }

# Загружаем обогащённый датасет
df = pd.read_pickle('enriched_sarcasm_pooled.pkl')
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['sarcasm'], random_state=42)

tokenizer = BertTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')
train_ds = SarcasmDataset(train_df, tokenizer)
eval_ds  = SarcasmDataset(test_df,  tokenizer)

bert = BertModel.from_pretrained('DeepPavlov/rubert-base-cased')

model = SarcasmClassifier(
    bert_model=bert,
    sem_dim=len(df['sem_pooled'].iloc[0]),
    num_classes=2
)

training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,

    # частота в шагах
    logging_steps=50,
    eval_steps=500,
    save_steps=500,
    save_total_limit=1,

    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics
)

# Тренировка и оценка
trainer.train()
metrics = trainer.evaluate()
print(metrics)

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Step,Training Loss
50,0.345700
100,0.173600
150,0.123200
200,0.107200
250,0.091200
300,0.076400
350,0.091800
400,0.072100
450,0.081400
500,0.059700


{'eval_loss': 0.11590506136417389, 'eval_accuracy': 0.963914707490432, 'eval_f1_macro': 0.9325707506066252, 'eval_runtime': 3.1569, 'eval_samples_per_second': 579.367, 'eval_steps_per_second': 9.186, 'epoch': 3.0}


In [16]:
'''import torch
import torch.nn as nn
from transformers import BertModel, BertConfig

class CustomBertClassifier(nn.Module):
    def __init__(self,
                 pretrained_model_name: str = 'bert-base-uncased',
                 num_labels: int = 2,
                 hidden_dim: int = 768,
                 dropout_prob: float = 0.1):
        super().__init__()
        # Базовая модель BERT без головы для маскированного языка
        self.bert = BertModel.from_pretrained(pretrained_model_name)

        # Кастомная голова
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_prob),
            nn.Linear(self.bert.config.hidden_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self,
                input_ids: torch.LongTensor,
                attention_mask: torch.Tensor = None,
                token_type_ids: torch.Tensor = None,
                labels: torch.LongTensor = None):
        # Получаем выходы из берта
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )
        # Выбираем pooled_output
        pooled_output = outputs.pooler_output

        # Передаём через свою голову
        logits = self.classifier(pooled_output)

        # Если есть метки - считаем loss
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
            return {
                'loss': loss,
                'logits': logits
            }
        return {'logits': logits}'''

In [17]:
'''import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import BertTokenizer, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from dataclasses import dataclass

combined_df = abc
combined_df['sarcasm'] = combined_df['sarcasm'].astype(int)

train_df, test_df = train_test_split(
    combined_df,
    test_size=0.2,
    stratify=combined_df['sarcasm'],
    random_state=42
)

# Токенизация
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

@dataclass
class NERFeatures:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    token_type_ids: torch.Tensor
    labels: torch.Tensor

class SarcasmDataset(TorchDataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['text'].tolist()
        self.labels = df['sarcasm'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        enc = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'token_type_ids': enc.get('token_type_ids', torch.zeros_like(enc['input_ids'])).squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Создаем модели
train_dataset = SarcasmDataset(train_df, tokenizer)
test_dataset  = SarcasmDataset(test_df, tokenizer)

# Загрузка модели
model = CustomBertClassifier(
    pretrained_model_name='bert-base-uncased',
    num_labels=2,
    hidden_dim=768,
    dropout_prob=0.1
)

# Метрики

def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='macro')
    }

# Параметры
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,

    # частота в шагах
    logging_steps=50,
    eval_steps=500,
    save_steps=500,
    save_total_limit=1,

    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
)

# Трейнер
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
results = trainer.evaluate()
print(results)
# Save the best model
trainer.save_model('./best_model')'''


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Step,Training Loss
50,0.486000


KeyboardInterrupt: 